In [2]:
import pandas as pd
import sqlite3

df = pd.read_csv('C:\\Users\\Admin\\personal-finance-analysis\\data\\personal_transactions.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Month']   = df['Date'].dt.strftime('%Y-%m')
df['Year']    = df['Date'].dt.year
df['Weekday'] = df['Date'].dt.day_name()

df_clean = df[df['Category'] != 'Credit Card Payment'].copy()
expenses = df_clean[df_clean['Transaction Type'] == 'debit'].copy()
income   = df_clean[df_clean['Transaction Type'] == 'credit'].copy()

# Загружаем в SQLite
conn = sqlite3.connect(':memory:')
expenses.to_sql('expenses', conn, index=False, if_exists='replace')
income.to_sql('income',   conn, index=False, if_exists='replace')

46

Query 1: Top of categories with budget share

In [3]:
q1 = """
SELECT 
    Category,
    ROUND(SUM(Amount), 2) AS total,
    ROUND(SUM(Amount) * 100.0 / (SELECT SUM(Amount) FROM expenses), 1) AS share_pct
FROM expenses
GROUP BY Category
ORDER BY total DESC
"""

print(pd.read_sql(q1, conn))

                  Category     total  share_pct
0          Mortgage & Rent  24754.50       39.3
1         Home Improvement  19092.87       30.3
2                Groceries   2795.21        4.4
3                Utilities   2776.00        4.4
4              Restaurants   2613.02        4.1
5                 Shopping   1973.24        3.1
6               Gas & Fuel   1715.17        2.7
7             Mobile Phone   1680.40        2.7
8                 Internet   1570.88        2.5
9           Auto Insurance   1350.00        2.1
10  Electronics & Software    719.00        1.1
11          Alcohol & Bars    539.13        0.9
12                 Haircut    378.00        0.6
13               Fast Food    330.63        0.5
14                   Music    224.49        0.4
15           Movies & DVDs    222.19        0.4
16            Coffee Shops    115.54        0.2
17              Television    104.78        0.2
18           Food & Dining     77.75        0.1
19           Entertainment      9.62    

Query 2: Average monthly expenses by category

In [4]:
q2 = """
SELECT 
    Category,
    ROUND(AVG(monthly_total), 2) AS avg_per_month
FROM (
    SELECT 
        Category,
        Month,
        SUM(Amount) AS monthly_total
    FROM expenses
    GROUP BY Category, Month
)
GROUP BY Category
ORDER BY avg_per_month DESC
"""

print(pd.read_sql(q2, conn))

                  Category  avg_per_month
0         Home Improvement        1363.78
1          Mortgage & Rent        1178.79
2   Electronics & Software         239.67
3                Groceries         133.11
4                Utilities         132.19
5              Restaurants         124.43
6                 Shopping          93.96
7               Gas & Fuel          81.67
8             Mobile Phone          80.02
9           Auto Insurance          75.00
10                Internet          74.80
11           Food & Dining          38.88
12               Fast Food          36.74
13          Alcohol & Bars          35.94
14                 Haircut          29.08
15           Movies & DVDs          18.52
16              Television          13.10
17                   Music          10.69
18           Entertainment           9.62
19            Coffee Shops           8.25


Query 3: Monthly balance

In [5]:
q3 = """
SELECT 
    e.Month,
    ROUND(e.total_expenses, 2) AS expenses,
    ROUND(i.total_income, 2)   AS income,
    ROUND(i.total_income - e.total_expenses, 2) AS balance
FROM (
    SELECT Month, SUM(Amount) AS total_expenses
    FROM expenses
    GROUP BY Month
) e
JOIN (
    SELECT Month, SUM(Amount) AS total_income
    FROM income
    GROUP BY Month
) i ON e.Month = i.Month
ORDER BY e.Month
"""

print(pd.read_sql(q3, conn))

      Month  expenses  income  balance
0   2018-01   2066.65  4000.0  1933.35
1   2018-02   2089.44  4000.0  1910.56
2   2018-03   2178.66  6000.0  3821.34
3   2018-04   2862.66  4000.0  1137.34
4   2018-05  10507.56  4000.0 -6507.56
5   2018-06   2115.05  4000.0  1884.95
6   2018-07   2302.64  4000.0  1697.36
7   2018-08   1992.68  6000.0  4007.32
8   2018-09   2314.74  4000.0  1685.26
9   2018-10   2242.37  4000.0  1757.63
10  2018-11   2118.15  4000.0  1881.85
11  2018-12   2652.94  4000.0  1347.06
12  2019-01   1736.43  4000.0  2263.57
13  2019-02   1954.60  4000.0  2045.40
14  2019-03   2142.59  6000.0  3857.41
15  2019-04   1966.87  4000.0  2033.13
16  2019-05   2253.41  4000.0  1746.59
17  2019-06  10912.38  4000.0 -6912.38
18  2019-07   2551.12  4500.0  1948.88
19  2019-08   2043.20  6750.0  4706.80
20  2019-09   2038.28  4500.0  2461.72


Query 4: Top-5 most expensive transactions

In [6]:
q4 = """
SELECT 
    Date,
    Description,
    Category,
    Amount
FROM expenses
ORDER BY Amount DESC
LIMIT 5
"""

print(pd.read_sql(q4, conn))

                  Date              Description          Category   Amount
0  2019-06-20 00:00:00  Mike's Construction Co.  Home Improvement  9200.00
1  2018-05-11 00:00:00  Mike's Construction Co.  Home Improvement  8000.00
2  2018-01-02 00:00:00         Mortgage Payment   Mortgage & Rent  1247.44
3  2018-02-02 00:00:00         Mortgage Payment   Mortgage & Rent  1247.44
4  2018-03-02 00:00:00         Mortgage Payment   Mortgage & Rent  1247.44


Query 5: Expenses by accounts

In [7]:
q5 = """
SELECT 
    "Account Name",
    ROUND(SUM(Amount), 2) AS total,
    COUNT(*) AS transactions,
    ROUND(AVG(Amount), 2) AS avg_transaction
FROM expenses
GROUP BY "Account Name"
ORDER BY total DESC
"""

print(pd.read_sql(q5, conn))

    Account Name     total  transactions  avg_transaction
0       Checking  49456.78           147           336.44
1  Platinum Card   8996.31           324            27.77
2    Silver Card   4589.33           146            31.43
